# Week 7 — Delta Lake MERGE Implementation
**Celebal Technologies | Data Engineer Internship**

Objective: perform incremental data processing (an upsert) using Delta Lake — load a customer master table, clean it, merge in an incremental batch of new + updated customers, and validate the result.

This notebook uses the official `deltalake` Python library (delta-rs) — real Delta Lake format, real transaction log, real MERGE — with no Spark/Java/Databricks required. A PySpark-equivalent reference for Databricks is included at the end.

```bash
pip install deltalake pandas pyarrow
```

## Step 0 — Generate `customer_master.csv` and `customer_incremental.csv`

(In a real assignment you'd download the Kaggle Superstore-style dataset; here we generate a small, reproducible customer dataset since the assignment's own data is 'a source dataset' + 'a simulated incremental batch' — synthetic data with a fixed seed makes the MERGE results reproducible.)

In [1]:
import random
import pandas as pd

random.seed(7)

CITIES = ["Delhi", "Mumbai", "Bengaluru", "Jaipur", "Hyderabad", "Pune", "Chennai", "Kolkata"]
STATUSES = ["Active", "Inactive", "Trial"]
FIRST_NAMES = ["Rahul", "Amit", "Priya", "Sneha", "Vikram", "Anjali", "Rohan", "Neha",
               "Karan", "Pooja", "Arjun", "Divya", "Manish", "Kavya", "Suresh", "Meera"]
LAST_NAMES = ["Sharma", "Verma", "Gupta", "Iyer", "Nair", "Reddy", "Singh", "Kapoor"]

N_MASTER = 80

# ---------------------------------------------------------------------------
# customer_master.csv — the existing / current data
# ---------------------------------------------------------------------------
rows = []
for i in range(1, N_MASTER + 1):
    cid = f"CUST{i:04d}"
    name = f"{random.choice(FIRST_NAMES)} {random.choice(LAST_NAMES)}"
    city = random.choice(CITIES)
    email = f"{name.lower().replace(' ', '.')}{i}@example.com"
    status = random.choice(STATUSES)
    signup_date = (pd.Timestamp("2023-01-01") + pd.Timedelta(days=random.randint(0, 700))).strftime("%Y-%m-%d")
    rows.append([cid, name, city, email, status, signup_date])

df_master = pd.DataFrame(rows, columns=["customer_id", "name", "city", "email", "status", "signup_date"])

# --- inject bad data for the cleaning step (Step 2) ---
missing_idx = df_master.sample(5, random_state=1).index
df_master.loc[missing_idx[:3], "email"] = None
df_master.loc[missing_idx[3:], "city"] = None

dupes = df_master.sample(4, random_state=2)
df_master = pd.concat([df_master, dupes], ignore_index=True)

df_master.to_csv("../data/customer_master.csv", index=False)

# ---------------------------------------------------------------------------
# customer_incremental.csv — new + updated records for the MERGE step
# ---------------------------------------------------------------------------
clean_master = df_master.drop_duplicates(subset=["customer_id"], keep="first")
existing_ids = clean_master["customer_id"].tolist()

# 12 existing customers get an UPDATE (status change and/or city change —
# simulates a customer moving city or upgrading from Trial to Active)
update_ids = random.sample(existing_ids, 12)
update_rows = []
for cid in update_ids:
    orig = clean_master[clean_master["customer_id"] == cid].iloc[0]
    new_city = random.choice([c for c in CITIES if c != orig["city"]])
    new_status = random.choice([s for s in STATUSES if s != orig["status"]])
    update_rows.append([
        cid, orig["name"], new_city, orig["email"], new_status,
        orig["signup_date"],
    ])

# 8 brand-new customers (INSERT)
new_rows = []
for j in range(1, 9):
    cid = f"CUST{N_MASTER + j:04d}"
    name = f"{random.choice(FIRST_NAMES)} {random.choice(LAST_NAMES)}"
    city = random.choice(CITIES)
    email = f"{name.lower().replace(' ', '.')}{N_MASTER + j}@example.com"
    status = random.choice(STATUSES)
    signup_date = pd.Timestamp("2025").strftime("%Y-%m-%d")
    new_rows.append([cid, name, city, email, status, signup_date])

df_incremental = pd.DataFrame(
    update_rows + new_rows,
    columns=["customer_id", "name", "city", "email", "status", "signup_date"],
)
df_incremental.to_csv("../data/customer_incremental.csv", index=False)

print(f"customer_master.csv      : {len(df_master)} rows ({len(clean_master)} unique customer_id)")
print(f"customer_incremental.csv : {len(df_incremental)} rows "
      f"({len(update_ids)} updates to existing customers, {len(new_rows)} new customers)")
print(f"\nSample updates (customer_id, old -> new city/status):")
for cid in update_ids[:5]:
    orig = clean_master[clean_master["customer_id"] == cid].iloc[0]
    upd = df_incremental[df_incremental["customer_id"] == cid].iloc[0]
    print(f"  {cid}: {orig['city']}/{orig['status']}  ->  {upd['city']}/{upd['status']}")

customer_master.csv      : 84 rows (80 unique customer_id)
customer_incremental.csv : 20 rows (12 updates to existing customers, 8 new customers)

Sample updates (customer_id, old -> new city/status):
  CUST0035: Mumbai/Active  ->  Chennai/Trial
  CUST0006: Delhi/Active  ->  Mumbai/Trial
  CUST0024: Chennai/Trial  ->  Delhi/Active
  CUST0080: Mumbai/Active  ->  Hyderabad/Inactive
  CUST0017: Mumbai/Active  ->  Jaipur/Inactive


## Setup — imports & paths

In [2]:
import shutil
import pandas as pd
from deltalake import DeltaTable, write_deltalake

DATA_DIR = "../data"
TABLE_PATH = "../delta_table/customers"


def section(title):
    print(f"\n{'=' * 70}\n{title}\n{'=' * 70}")

## STEP 1: Load the source dataset into a Delta table

In [3]:
section("STEP 1 — Load source dataset into a Delta table")

df_master_raw = pd.read_csv(f"{DATA_DIR}/customer_master.csv")
print(f"Read customer_master.csv: {len(df_master_raw)} rows")

# Fresh table for this run
shutil.rmtree(TABLE_PATH, ignore_errors=True)
write_deltalake(TABLE_PATH, df_master_raw, mode="overwrite")

dt = DeltaTable(TABLE_PATH)
print(f"Delta table created at {TABLE_PATH}")
print(f"Delta log version after initial load: {dt.version()}")
print(f"Row count in Delta table: {len(dt.to_pandas())}")


STEP 1 — Load source dataset into a Delta table
Read customer_master.csv: 84 rows
Delta table created at ../delta_table/customers
Delta log version after initial load: 0


Row count in Delta table: 84


## STEP 2: Perform basic cleaning (nulls, duplicates) directly on the Delta table

In [4]:
section("STEP 2 — Basic cleaning: nulls + duplicates")

df = dt.to_pandas()
print("Null counts before cleaning:")
print(df.isnull().sum().to_string())

before = len(df)
df = df.drop_duplicates(subset=["customer_id"], keep="first")
print(f"\nDropped {before - len(df)} duplicate customer_id rows")

df["email"] = df["email"].fillna("unknown@example.com")
df["city"] = df["city"].fillna("Unknown")
print("Filled missing email -> 'unknown@example.com', missing city -> 'Unknown'")

# Overwrite the Delta table with the cleaned data (new Delta version, old
# version still recoverable via time travel -- this is not a destructive
# operation at the storage layer, just a new commit in the transaction log)
write_deltalake(TABLE_PATH, df, mode="overwrite")
dt = DeltaTable(TABLE_PATH)
print(f"\nDelta log version after cleaning commit: {dt.version()}")
print(f"Row count after cleaning: {len(dt.to_pandas())}")


STEP 2 — Basic cleaning: nulls + duplicates
Null counts before cleaning:
customer_id    0
name           0
city           2
email          3
status         0
signup_date    0

Dropped 4 duplicate customer_id rows
Filled missing email -> 'unknown@example.com', missing city -> 'Unknown'

Delta log version after cleaning commit: 1
Row count after cleaning: 80


## STEP 3: Create/load the incremental dataset (new + updated records)

In [5]:
section("STEP 3 — Load the incremental dataset")

df_incremental = pd.read_csv(f"{DATA_DIR}/customer_incremental.csv")
print(f"Read customer_incremental.csv: {len(df_incremental)} rows")

existing_ids = set(dt.to_pandas()["customer_id"])
incoming_ids = set(df_incremental["customer_id"])
n_updates = len(incoming_ids & existing_ids)
n_inserts = len(incoming_ids - existing_ids)
print(f"Of these: {n_updates} match an existing customer_id (-> UPDATE), "
      f"{n_inserts} are new (-> INSERT)")


STEP 3 — Load the incremental dataset
Read customer_incremental.csv: 20 rows
Of these: 12 match an existing customer_id (-> UPDATE), 8 are new (-> INSERT)


## STEP 4: Apply the MERGE operation — SCD Type 1 (overwrite in place)

In [6]:
section("STEP 4 — MERGE (SCD Type 1: update matched rows in place, insert new ones)")

pre_merge_count = len(dt.to_pandas())

(
    dt.merge(
        source=df_incremental,
        predicate="t.customer_id = s.customer_id",
        source_alias="s",
        target_alias="t",
    )
    .when_matched_update_all()
    .when_not_matched_insert_all()
    .execute()
)

print(f"MERGE complete. Delta log version: {dt.version()}")
post_merge_count = len(dt.to_pandas())
print(f"Row count before merge: {pre_merge_count}  ->  after merge: {post_merge_count} "
      f"(+{post_merge_count - pre_merge_count} net new rows, matches the {n_inserts} inserts)")


STEP 4 — MERGE (SCD Type 1: update matched rows in place, insert new ones)


MERGE complete. Delta log version: 2
Row count before merge: 80  ->  after merge: 88 (+8 net new rows, matches the 8 inserts)


## STEP 5: Validate results

In [7]:
section("STEP 5 — Validate the merged table")

final_df = dt.to_pandas()

# 5a. Total row count should equal cleaned-master-unique-ids + new inserts
expected = len(existing_ids) + n_inserts
print(f"Total row count: {len(final_df)}  (expected {len(existing_ids)} existing + {n_inserts} new = {expected})"
      f"  -> {'PASS' if len(final_df) == expected else 'FAIL'}")

# 5b. No duplicate customer_id
dup_count = final_df["customer_id"].duplicated().sum()
print(f"Duplicate customer_id rows: {dup_count}  -> {'PASS' if dup_count == 0 else 'FAIL'}")

# 5c. Spot-check that updates were actually applied
sample_update_id = df_incremental.iloc[0]["customer_id"]
expected_row = df_incremental[df_incremental["customer_id"] == sample_update_id].iloc[0]
actual_row = final_df[final_df["customer_id"] == sample_update_id].iloc[0]
update_ok = (expected_row["city"] == actual_row["city"]) and (expected_row["status"] == actual_row["status"])
print(f"Spot check {sample_update_id}: expected city/status "
      f"{expected_row['city']}/{expected_row['status']}, got {actual_row['city']}/{actual_row['status']}"
      f"  -> {'PASS' if update_ok else 'FAIL'}")

# 5d. Spot-check that a new customer was actually inserted
new_ids_actual = set(df_incremental["customer_id"]) - existing_ids
sample_new_id = sorted(new_ids_actual)[0]
insert_ok = sample_new_id in set(final_df["customer_id"])
print(f"Spot check new customer {sample_new_id} present in final table -> {'PASS' if insert_ok else 'FAIL'}")


STEP 5 — Validate the merged table
Total row count: 88  (expected 80 existing + 8 new = 88)  -> PASS
Duplicate customer_id rows: 0  -> PASS
Spot check CUST0035: expected city/status Chennai/Trial, got Chennai/Trial  -> PASS
Spot check new customer CUST0081 present in final table -> PASS


## STEP 6: Display the final merged Delta table + summary

In [8]:
section("STEP 6 — Final merged Delta table")

print(final_df.sort_values("customer_id").head(10).to_string(index=False))
print(f"\n... {len(final_df)} rows total")

section("Delta transaction log (dt.history()) — every commit made in this run")
history = dt.history()
for h in history:
    op = h.get("operation")
    ver = h.get("version")
    metrics = h.get("operationMetrics", {})
    print(f"version {ver:>2}: {op:<10} {metrics}")

section("SUMMARY")
print(f"""
- Loaded {len(df_master_raw)} raw master rows into a Delta table (version 0).
- Cleaning removed {before - len(df)} duplicate rows and filled {df_master_raw.isnull().sum().sum()} nulls,
  committed as a new Delta version.
- Merged {len(df_incremental)} incremental records: {n_updates} matched existing
  customers and were updated in place, {n_inserts} were new and inserted.
- Final table has {len(final_df)} rows, 0 duplicate customer_id values.
- Every step is a separate, auditable commit in the Delta transaction log
  (see dt.history() above) — this is the core benefit Delta Lake adds over
  plain Parquet/CSV: ACID commits and the ability to time-travel back to
  any prior version if a MERGE ever needs to be undone.
""")


STEP 6 — Final merged Delta table


customer_id         name      city                     email   status signup_date
   CUST0001  Arjun Gupta   Chennai  arjun.gupta1@example.com    Trial  2023-02-19
   CUST0002  Priya Verma      Pune  priya.verma2@example.com    Trial  2023-03-01
   CUST0003 Rohan Sharma    Mumbai rohan.sharma3@example.com Inactive  2024-03-04
   CUST0004   Priya Iyer    Mumbai   priya.iyer4@example.com    Trial  2024-03-10
   CUST0005   Amit Verma    Jaipur   amit.verma5@example.com    Trial  2024-10-04
   CUST0006   Amit Singh    Mumbai   amit.singh6@example.com    Trial  2023-02-17
   CUST0007  Vikram Nair   Chennai  vikram.nair7@example.com   Active  2024-07-07
   CUST0008   Sneha Nair Bengaluru   sneha.nair8@example.com   Active  2024-08-18
   CUST0009  Rohan Reddy    Mumbai  rohan.reddy9@example.com    Trial  2023-03-06
   CUST0010    Amit Iyer   Kolkata   amit.iyer10@example.com    Trial  2024-06-28

... 88 rows total

Delta transaction log (dt.history()) — every commit made in this run
version  

## Bonus — SCD Type 2 (history-preserving MERGE)

Week 7 - Bonus: SCD Type 2 MERGE (history-preserving upsert)
===============================================================
The core assignment (delta_merge_pipeline.py) implements SCD Type 1:
an update simply overwrites the old value, no history kept. The
suggested screenshot folders (scd1/, scd2/) imply the instructor also
wants a Type 2 demonstration, so this script builds that as a bonus,
on a separate Delta table so it doesn't interfere with the Type 1
deliverable.

SCD Type 2 keeps every version of a row instead of overwriting it:
    - is_current   : True for the currently-active version of a customer
    - effective_date: when this version became active
    - end_date      : when this version was superseded (None if current)

Two-phase MERGE, the standard simplified pattern:
    Phase A (expire) : for customer_ids present in the incremental batch
                        AND currently active, MERGE sets is_current=False,
                        end_date=today on their *old* row.
    Phase B (insert)  : append a brand-new row for every incoming record
                        (both updates and new customers) with
                        is_current=True, effective_date=today, end_date=None.

In [9]:
import shutil
import pandas as pd
from deltalake import DeltaTable, write_deltalake

DATA_DIR = "../data"
TABLE_PATH = "../delta_table/customers_scd2"
TODAY = pd.Timestamp.today().strftime("%Y-%m-%d")


def section(title):
    print(f"\n{'=' * 70}\n{title}\n{'=' * 70}")


# ---------------------------------------------------------------------------
# Build the initial SCD2 table from the already-cleaned master data
# ---------------------------------------------------------------------------
section("Build initial SCD2 table from cleaned master data")

df_master = pd.read_csv(f"{DATA_DIR}/customer_master.csv").drop_duplicates(
    subset=["customer_id"], keep="first"
)
df_master["email"] = df_master["email"].fillna("unknown@example.com")
df_master["city"] = df_master["city"].fillna("Unknown")

df_master["effective_date"] = df_master["signup_date"]
df_master["end_date"] = pd.array([None] * len(df_master), dtype="string")
df_master["is_current"] = True

shutil.rmtree(TABLE_PATH, ignore_errors=True)
write_deltalake(TABLE_PATH, df_master, mode="overwrite")
dt = DeltaTable(TABLE_PATH)
print(f"Initial SCD2 table: {len(df_master)} rows, all is_current=True, version {dt.version()}")


# ---------------------------------------------------------------------------
# Phase A — expire the old version of every customer in the incremental batch
# ---------------------------------------------------------------------------
section("Phase A — expire old rows for customers being updated")

df_incremental = pd.read_csv(f"{DATA_DIR}/customer_incremental.csv")
expire_source = df_incremental[["customer_id"]].copy()
expire_source["end_date"] = TODAY

(
    dt.merge(
        source=expire_source,
        predicate="t.customer_id = s.customer_id AND t.is_current = true",
        source_alias="s",
        target_alias="t",
    )
    .when_matched_update(
        updates={"is_current": "false", "end_date": "s.end_date"}
    )
    .execute()
)
print(f"Expire commit done. Delta version: {dt.version()}")
expired_count = (dt.to_pandas()["is_current"] == False).sum()  # noqa: E712
print(f"Rows now marked is_current=False: {expired_count} "
      f"(expected: customers in the incremental batch that already existed)")


# ---------------------------------------------------------------------------
# Phase B — append a fresh current-version row for every incoming record
# ---------------------------------------------------------------------------
section("Phase B — insert new current-version rows")

new_versions = df_incremental.copy()
new_versions["effective_date"] = TODAY
new_versions["end_date"] = pd.array([None] * len(new_versions), dtype="string")
new_versions["is_current"] = True

write_deltalake(TABLE_PATH, new_versions, mode="append")
dt = DeltaTable(TABLE_PATH)
print(f"Appended {len(new_versions)} new current-version rows. Delta version: {dt.version()}")


# ---------------------------------------------------------------------------
# Validate
# ---------------------------------------------------------------------------
section("Validate SCD2 table")

final = dt.to_pandas()
current_view = final[final["is_current"] == True]  # noqa: E712

print(f"Total historical rows (all versions): {len(final)}")
print(f"Current rows (is_current=True): {len(current_view)}")
print(f"Duplicate customer_id among CURRENT rows: "
      f"{current_view['customer_id'].duplicated().sum()}  -> should be 0")

# Spot check: a customer that was updated should now have 2 rows (1 expired, 1 current)
sample_id = df_incremental.iloc[0]["customer_id"]
versions = final[final["customer_id"] == sample_id].sort_values("effective_date")
print(f"\nVersion history for {sample_id} ({len(versions)} version(s)):")
print(versions[["customer_id", "city", "status", "effective_date", "end_date", "is_current"]].to_string(index=False))


section("SCD1 vs SCD2 — the difference this demonstrates")
print("""
SCD Type 1 (delta_merge_pipeline.py): the old row is overwritten in place.
    After the MERGE you can only ever see the customer's LATEST city/status —
    the fact that CUST0035 used to live in Mumbai is gone.

SCD Type 2 (this script): the old row is kept, just marked is_current=False
    with an end_date. A new row is added for the new values. The full
    history of every change a customer ever had is queryable at any time —
    at the cost of the table growing with every update instead of staying
    the same size.
""")


Build initial SCD2 table from cleaned master data
Initial SCD2 table: 80 rows, all is_current=True, version 0

Phase A — expire old rows for customers being updated
Expire commit done. Delta version: 1


Rows now marked is_current=False: 12 (expected: customers in the incremental batch that already existed)

Phase B — insert new current-version rows
Appended 20 new current-version rows. Delta version: 2

Validate SCD2 table


Total historical rows (all versions): 100
Current rows (is_current=True): 88
Duplicate customer_id among CURRENT rows: 0  -> should be 0

Version history for CUST0035 (2 version(s)):
customer_id    city status effective_date   end_date  is_current
   CUST0035  Mumbai Active     2024-09-20 2026-08-03       False
   CUST0035 Chennai  Trial     2026-08-03        NaN        True

SCD1 vs SCD2 — the difference this demonstrates

SCD Type 1 (delta_merge_pipeline.py): the old row is overwritten in place.
    After the MERGE you can only ever see the customer's LATEST city/status —
    the fact that CUST0035 used to live in Mumbai is gone.

SCD Type 2 (this script): the old row is kept, just marked is_current=False
    with an end_date. A new row is added for the new values. The full
    history of every change a customer ever had is queryable at any time —
    at the cost of the table growing with every update instead of staying
    the same size.



## Appendix — PySpark / Databricks equivalent (reference only)

Not an executable cell in this notebook — `configure_spark_with_delta_pip()` needs to download the `delta-spark` jar from Maven Central at runtime, which this build environment doesn't have network access to. On your own machine or in Databricks (both have normal internet access), this code runs as-is and produces the same MERGE result validated above.

```python
"""
PySpark + delta-spark equivalent (for Databricks / a machine with full
internet access to Maven Central). NOT executed in this notebook —
provided as a reference so the exact same MERGE logic above can be
reproduced on a Spark cluster. Requires: pip install delta-spark, and
a SparkSession built with configure_spark_with_delta_pip().
"""
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable as SparkDeltaTable

builder = (
    SparkSession.builder.appName("delta_merge")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

# Step 1: load into Delta
df_master = spark.read.option("header", True).csv("../data/customer_master.csv")
df_master.write.format("delta").mode("overwrite").save("/tmp/customers_delta")

# Step 2: clean (nulls + duplicates) -- same idea as the pandas version above
clean = df_master.dropDuplicates(["customer_id"]).fillna({"email": "unknown@example.com", "city": "Unknown"})
clean.write.format("delta").mode("overwrite").save("/tmp/customers_delta")

# Step 3/4: MERGE the incremental batch
incremental = spark.read.option("header", True).csv("../data/customer_incremental.csv")
target = SparkDeltaTable.forPath(spark, "/tmp/customers_delta")

(
    target.alias("t")
    .merge(incremental.alias("s"), "t.customer_id = s.customer_id")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

# Step 6: same validation queries as above, just via Spark SQL / DataFrame API
target.toDF().show()
```